In [2]:
import pandas as pd
from pathlib import Path
processed_dir = Path.cwd().parent / "data" / "processed"
raw_dir = Path.cwd().parent / "data" / "raw"

In [ ]:
nav_df = pd.read_csv(raw_dir / "02_nav_history.csv")
nav_df['date'] = pd.to_datetime(nav_df['date'], format='mixed', dayfirst=True)
nav_df = nav_df.drop_duplicates(subset=['amfi_code', 'date'])
nav_df = nav_df[nav_df['nav'] > 0]
nav_df.set_index('date', inplace=True)
cleaned_nav = nav_df.groupby('amfi_code')['nav'].resample('D').ffill().reset_index()
cleaned_nav = cleaned_nav.sort_values(['amfi_code', 'date']).reset_index(drop=True)
cleaned_nav.to_csv(processed_dir / "02_nav_history_cleaned.csv", index=False)

In [5]:
investor_df = pd.read_csv(raw_dir / "08_investor_transactions.csv")
investor_df['transaction_date'] = pd.to_datetime(investor_df['transaction_date'], format='mixed', dayfirst=True)
investor_df = investor_df[investor_df['amount_inr'] > 0]
investor_df['transaction_type'] = investor_df['transaction_type'].str.upper().str.strip().str.replace('.', '',regex=False)
investor_df['kyc_status'] = investor_df['kyc_status'].str.upper().str.strip()
investor_df.to_csv(processed_dir / "08_investor_transactions_cleaned.csv", index=False)

In [8]:
performance_df = pd.read_csv(raw_dir / "07_scheme_performance.csv")
print("Columns:", performance_df.columns.tolist())
return_columns = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']
for col in return_columns:
    if col in performance_df.columns:
        performance_df[col] = pd.to_numeric(performance_df[col], errors='coerce')
valid_expense = (performance_df['expense_ratio_pct'] >= 0.1) & (performance_df['expense_ratio_pct'] <= 2.5)
print(f"Invalid expense ratio entries: { (~valid_expense).sum() } out of {len(performance_df)}")
performance_df = performance_df[valid_expense].reset_index(drop=True)
performance_df.to_csv(processed_dir / "07_scheme_performance_cleaned.csv", index=False)
print("Data cleaning completed. Cleaned files saved to:", processed_dir)

Columns: ['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']
Invalid expense ratio entries: 0 out of 40
Data cleaning completed. Cleaned files saved to: /Users/Shourya_1/Capstone_Proj/data/processed


In [17]:
all_raw_files = list(raw_dir.glob("*.csv"))
already_cleaned = [
    "02_nav_history.csv", 
    "08_investor_transactions.csv", 
    "07_scheme_performance.csv"
]
for file in all_raw_files:
    if file.name not in already_cleaned:
        print(f"Copying {file.name} to processed directory.")
        df = pd.read_csv(file)
        df = df.dropna(how='all').drop_duplicates()
        new_name = file.stem + "_cleaned.csv"
        df.to_csv(processed_dir / new_name, index=False)


Copying 01_fund_master.csv to processed directory.
Copying 04_monthly_sip_inflows.csv to processed directory.
Copying 10_benchmark_indices.csv to processed directory.
Copying 09_portfolio_holdings.csv to processed directory.
Copying 05_category_inflows.csv to processed directory.
Copying 03_aum_by_fund_house.csv to processed directory.
Copying 06_industry_folio_count.csv to processed directory.


In [ ]:
import sqlite3
db_path = Path.cwd().parent / "data" / "db" / "bluestock_mf.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
schema = """

CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    fund_house TEXT,
    scheme_name TEXT,
    category TEXT,
    sub_category TEXT,
    plan TEXT,
    risk_grade TEXT
);

CREATE TABLE IF NOT EXISTS dim_date (
    date TEXT PRIMARY KEY,
    year INTEGER,
    month INTEGER,
    day INTEGER,
    quarter INTEGER,
    is_weekend BOOLEAN
);

CREATE TABLE IF NOT EXISTS fact_nav (
    amfi_code INTEGER,
    date TEXT,
    nav REAL,
    PRIMARY KEY (amfi_code, date),
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (date) REFERENCES dim_date(date)
);

CREATE TABLE IF NOT EXISTS fact_transactions (
    investor_id TEXT,
    transaction_date TEXT,
    amfi_code INTEGER,
    transaction_type TEXT,
    amount_inr REAL,
    state TEXT,
    city TEXT,
    city_tier TEXT,
    age_group TEXT,
    gender TEXT,
    annual_income_lakh REAL,
    payment_mode TEXT,
    kyc_status TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (transaction_date) REFERENCES dim_date(date)
);

CREATE TABLE IF NOT EXISTS fact_performance (
    amfi_code INTEGER PRIMARY KEY,
    return_1yr_pct REAL,
    return_3yr_pct REAL,
    return_5yr_pct REAL,
    benchmark_3yr_pct REAL,
    alpha REAL,
    beta REAL,
    sharpe_ratio REAL,
    sortino_ratio REAL,
    std_dev_ann_pct REAL,
    max_drawdown_pct REAL,
    expense_ratio_pct REAL,
    morningstar_rating INTEGER,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

    CREATE TABLE IF NOT EXISTS fact_aum (
    amfi_code INTEGER PRIMARY KEY,
    aum_crore REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);
"""
cursor.executescript(schema)
conn.commit()
conn.close()
print("Database schema created successfully at ", db_path)

Database schema created successfully at  /Users/Shourya_1/Capstone_Proj/sql/bluestock_mf.db


In [ ]:
from sqlalchemy import create_engine
db_path = Path.cwd().parent / "data" / "db" / "bluestock_mf.db"
engine = create_engine(f'sqlite:///{db_path}')
processed_dir = Path.cwd().parent / "data" / "processed"
fund_df = pd.read_csv(processed_dir / "01_fund_master_cleaned.csv")
nav_df = pd.read_csv(processed_dir / "02_nav_history_cleaned.csv")
txn_df = pd.read_csv(processed_dir / "08_investor_transactions_cleaned.csv")
performance_df = pd.read_csv(processed_dir / "07_scheme_performance_cleaned.csv")
unique_dates = pd.to_datetime(nav_df['date'].unique())
dim_date_df = pd.DataFrame({
    'date': unique_dates.astype(str)})
dim_date_df['year'] = unique_dates.year
dim_date_df['month'] = unique_dates.month
dim_date_df['day'] = unique_dates.day
dim_date_df['quarter'] = unique_dates.quarter
dim_date_df['is_weekend'] = unique_dates.dayofweek >= 5
fact_aum = performance_df[['amfi_code', 'aum_crore']].copy()

dim_fund_df = fund_df[['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'risk_category']].rename(columns={'risk_category': 'risk_grade'})
fact_nav = nav_df[['amfi_code', 'date', 'nav']]
fact_transactions = txn_df[['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']]
fact_performance = performance_df[['amfi_code', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'expense_ratio_pct', 'morningstar_rating']]

tables = {
    'dim_fund': dim_fund_df,
    'dim_date': dim_date_df,
    'fact_nav': fact_nav,
    'fact_transactions': fact_transactions,
    'fact_performance': fact_performance,
    'fact_aum': fact_aum
}
for table_name, df in tables.items():
    try:
        df.to_sql(table_name, con=engine, if_exists='append', index=False)
        print(f"Inserted {len(df)} records into {table_name}")
    except Exception as e:
        print(f"Error inserting into {table_name}: {e}")

Inserted 40 records into dim_fund
Inserted 1799 records into dim_date
Inserted 71960 records into fact_nav
Inserted 32778 records into fact_transactions
Inserted 40 records into fact_performance
Inserted 40 records into fact_aum


In [3]:
from sqlalchemy import create_engine
db_path = Path.cwd().parent / "data" / "db" / "bluestock_mf.db"
engine = create_engine(f'sqlite:///{db_path}')
q1 = """
SELECT f.scheme_name, a.aum_crore
FROM dim_fund f
JOIN fact_aum a ON f.amfi_code = a.amfi_code
ORDER BY a.aum_crore DESC
LIMIT 5;
"""
print("--- Top 5 Funds by AUM ---")
display(pd.read_sql(q1, engine))
q2 = """
SELECT state, COUNT(investor_id) as total_transactions, SUM(amount_inr) as total_volume_inr
FROM fact_transactions
GROUP BY state
ORDER BY total_volume_inr DESC;
"""
print("\n--- Transactions by State ---")
display(pd.read_sql(q2, engine))

q3 = """
SELECT f.scheme_name, f.category, p.expense_ratio_pct
FROM dim_fund f
JOIN fact_performance p ON f.amfi_code = p.amfi_code
WHERE p.expense_ratio_pct < 1.0
ORDER BY p.expense_ratio_pct ASC;
"""
print("\n--- Funds with Expense Ratio < 1% ---")
display(pd.read_sql(q3, engine))


--- Top 5 Funds by AUM ---


,scheme_name,aum_crore
0,Mirae Asset Emerging Bluechip Fund - Regular -...,49046.0
1,Kotak Emerging Equity Fund - Regular - Growth,47469.0
2,Nippon India Small Cap Fund - Regular - Growth,43630.0
3,DSP Top 100 Equity Fund - Regular - Growth,41828.0
4,UTI Mid Cap Fund - Regular - Growth,41728.0



--- Transactions by State ---


,state,total_transactions,total_volume_inr
0,Punjab,2965,315780459.0
1,Tamil Nadu,2806,315177237.0
2,Madhya Pradesh,2931,308312493.0
3,Rajasthan,2577,298645822.0
4,Gujarat,2780,298358940.0
5,West Bengal,2748,297182514.0
6,Telangana,2718,290219284.0
7,Delhi,2677,289633404.0
8,Uttar Pradesh,2695,285368873.0
9,Haryana,2736,279634354.0



--- Funds with Expense Ratio < 1% ---


,scheme_name,category,expense_ratio_pct
0,Nippon India Gilt Securities Fund - Regular - ...,Debt,0.55
1,HDFC Short Term Debt Fund - Regular - Growth,Debt,0.56
2,Kotak Liquid Fund - Regular - Growth,Debt,0.60
3,SBI Bluechip Fund - Direct Plan - Growth,Equity,0.66
4,Nippon India Large Cap Fund - Direct - Growth,Equity,0.72
5,SBI Small Cap Fund - Direct Plan - Growth,Equity,0.72
6,ICICI Pru Liquid Fund - Regular - Growth,Debt,0.74
7,Axis Bluechip Fund - Direct - Growth,Equity,0.75
8,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,0.77
9,HDFC Mid-Cap Opportunities Fund - Direct - Growth,Equity,0.78
